In [ ]:
from periodic_c_problem import PeriodicCProblem
from regpy.util.operator_tests import test_derivative, test_adjoint
from regpy.solvers import Setting
from regpy.solvers.nonlinear import NewtonCG, IrgnmCG, Landweber, FISTA
from regpy.stoprules import Discrepancy,CountIterations, OptimalityCondStoppingNonConvex
from regpy.hilbert import L2, Hm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
n = 100
#f = np.zeros(100); f[0]=1.
f = np.ones(100)
F = PeriodicCProblem(n=100,f=f,k=10.)



In [ ]:
def print_distances(chat,cdag,udag,uhat):
    norm = setting.h_domain.norm
    conorm = setting.h_codomain.norm
    err =  norm(chat-cdag)/norm(cdag)
    L2err = np.linalg.norm(chat-cdag)/np.linalg.norm(cdag)
    imerr = conorm(uhat-udag)/conorm(udag)
    print('rel. distance reco H1/L2', err, L2err)
    print('rel. distance image space', imerr)
    return err, L2err, imerr

In [ ]:
pts = F.grid.coords[0]
cdag = (1.+0.3*np.sin(2*np.pi*pts))/(1.-0.5*np.cos(6*np.pi*pts))
udag = F(cdag)


In [ ]:
tikh_setting = Setting(F, 
                  Hm(index=1), 
                  L2,                   
                  data = udag, 
                  init = F.domain.ones(),
                  codomain_free=False,
                  regpar=1e-10)
tikh_setting_codomain_free = Setting(F, 
                  Hm(index=1), 
                  L2, 
                  data = udag, 
                  init = F.domain.ones(),
                  codomain_free=True,
                  regpar=1e-10)
tikh_setting.display_all_methods()
tikh_setting_codomain_free.display_all_methods()

In [ ]:
setting = Setting(F,  
                  Hm(index=1), 
                  L2,
                  data = udag, 
                  init = F.domain.ones(),
                  codomain_free=False,
                    )
setting.display_all_methods()
setting_codomain_free = Setting(F, 
                  Hm(index=1), 
                  L2,
                  data = udag, 
                  init = F.domain.ones(),
                  codomain_free=True,
                    )
setting_codomain_free.display_all_methods()

# test codomain-free vs standard

In [ ]:
chat, _ = tikh_setting.run('genFISTA',max_it=2,logging_level = 'DEBUG')
chat_cdf, _ = tikh_setting_codomain_free.run('genFISTA',max_it=2,logging_level = 'DEBUG')

err, L2err, _ = print_distances(chat,chat_cdf,udag,udag)
print(err,L2err)
#assert err<=1e-8

In [ ]:
chat, _ = setting.run('nlLW',max_it=2,stepsize = 2e6,logging_level = 'WARNING')
chat_cdf, _ = setting_codomain_free.run('nlLW',max_it=2,stepsize = 2e6,logging_level = 'DEBUG')

err, _, _ = print_distances(chat,chat_cdf,udag,udag)
assert err <=1e-8

for method in ['IRGNM_CG','LevMar']:
    chat, _ = setting.run(method,max_it=2,logging_level = 'WARNING')
    chat_cdf, _ = setting_codomain_free.run(method,max_it=2,logging_level = 'WARNING')

    print(method)
    err, _, _ = print_distances(chat,chat_cdf,udag,udag)
    assert err <=1e-10


# test convergence for exact data

In [ ]:
#chat, uhat = setting.run('NewtCGFroz',max_it=40, logging_level='DEBUG')
#print_distances(chat,cdag,uhat,udag)

In [ ]:
"""fig, [ax1,ax2,ax3] = plt.subplots(1,3,figsize = [12,4])
ax1.plot(pts,chat,label='c_NCG')
ax1.plot(pts,cdag,label='true c')
ax1.legend()
ax2.plot(pts,uhat,label='u_NCG')
ax2.plot(pts,udag,label='exact u')
ax2.legend()
ax3.plot(pts,uhat-udag,label= 'diff data')
ax3.legend()"""


In [ ]:
chat, uhat = setting.run('nlLW',max_it=400,ref_sol=cdag,logging_level='WARNING',)
print('\nnlLW')
err, errL2, imerr = print_distances(chat,cdag,uhat,udag)
assert errL2<=1e-1; assert imerr<=1e-2

chat, uhat = setting.run('genFISTA',max_it=100,ref_sol=cdag,logging_level='WARNING',
                         stepsize_selection_steps = [0])
print('\ngenFISTA')
err, errL2, imerr = print_distances(chat,cdag,uhat,udag)
assert errL2<=1e-1; assert imerr<=1e-2

chat, uhat = setting.run('IRGNM_CG',max_it=20,ref_sol=cdag,regpar = 1e-9, regpar_step = 1/3, logging_level='WARNING')
print('\nIRGNM_CG')
err, errL2, imerr = print_distances(chat,cdag,uhat,udag)
assert errL2<=1e-2; assert imerr<=5e-4

chat, uhat = setting.run('LevMar',max_it=15,ref_sol=cdag,regpar = 1e-8, regpar_step = 1/2, logging_level='WARNING')
print('\nLevMar')
err, errL2, imerr = print_distances(chat,cdag,uhat,udag)
assert errL2<=1e-2; assert imerr<5e-4

chat, uhat = setting.run('NewtonCG',max_it=25, ref_sol=cdag,logging_level='WARNING')
print('\nNewtonCG')
err, errL2, imerr = print_distances(chat,cdag,uhat,udag)
assert errL2<=2e-2; assert imerr<8e-3

#chat, uhat = setting.run('NewtCGFroz',max_it=20, ref_sol=cdag,logging_level='WARNING')
#print('\nNewtCGFroz')
#err, errL2, imerr = print_distances(chat,cdag,uhat,udag)
#assert errL2<=1e-1; assert imerr<1e-2

In [ ]:
from regpy.solvers.plotting import convergence_plots
convergence_plots(setting)

In [ ]:
tikh_setting.regpar = 1e-9
tikh_setting.set_stoprule(OptimalityCondStoppingNonConvex(tol=0.1),
                          method_names=['genFISTA'])
chat, uhat = tikh_setting.run('genFISTA',max_it=400,logging_level = 'DEBUG')
print_distances(chat,cdag,uhat,udag)

In [ ]:
#setting.set_method_params('TikhCG',{'reltolx':1/3})

In [ ]:
max_it = 3
cg_pars = {'reltolx':1e-8,
          'reltoly':1e-8,
           'all_tol_criteria':False
         }
setting.set_method_params('IRN',{'inner_method':'TikhCG',
                                 'inner_params': cg_pars,
                                 "inner_stoprule": lambda inner_setting, params: CountIterations(
        max_iterations=1000)
                                 }
                                 )
setting.set_method_parameters('IRGNM_CG',{'cg_pars':cg_pars})
cIRN, uIRN = setting.run('IRN',max_it=max_it,ref_sol=cdag,regpar = 1e-9, regpar_step = 1/3,
                         inner_method='TikhCG',
                         logging_level='INFO')
cIRGNM, uIRGNM = setting.run('IRGNM_CG',max_it=max_it,ref_sol=cdag,regpar = 1e-9, regpar_step = 1/3, 
                         logging_level='INFO')
err, errL2, imerr = print_distances(cIRGNM,cIRN,uIRGNM,uIRN)
assert err<1e-8
assert errL2<1e-8
assert imerr<1e-8
